# 01. LangChain tool call


> 랭체인에서도 tool 사용 가능

In [40]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-5.4-nano")

llm.invoke([HumanMessage("하이~")])

AIMessage(content='하이~ 😊  \n오늘 뭐 도와줄까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 9, 'total_tokens': 24, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DuulJvUbkmUbyZUVGqDauikMTrK6f', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f02b6-5c81-72c1-a174-9e8656cdb1de-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'output_tokens': 15, 'total_tokens': 24, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [41]:
from langchain_core.tools import tool

In [42]:
from datetime import datetime
import pytz

@tool # @tool 데코레이터를 사용하여 함수를 도구로 등록
def get_current_time(timezone: str, location: str) -> str:
    ### 단순한 주석이 아니라 랭체인에 이 함수의 기능과 입력값, 사용 방법을 알려주는 문서화 문자열
    """ 현재 시각을 반환하는 함수 

    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨
    """
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재시각 {now} ' # 타임존, 지역명, 현재시각을 문자열로 반환
    print(location_and_local_time)
    return location_and_local_time


In [43]:
# 도구를 tools 리스트에 추가하고, tool_dict에도 추가
tools = [get_current_time,]
tool_dict = {"get_current_time": get_current_time,}

In [44]:
# 도구를 모델에 바인딩: 모델에 도구를 바인딩하면, 도구를 사용하여 llm 답변을 생성할 수 있음
llm_with_tools = llm.bind_tools(tools)

In [45]:
from langchain_core.messages import SystemMessage

# (4) 사용자의 질문과 tools 사용하여 llm 답변 생성
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("서울은 지금 몇시야?"),
]

# (5) llm_with_tools를 사용하여 사용자의 질문에 대한 llm 답변 생성
response = llm_with_tools.invoke(messages)
messages.append(response)

In [46]:
messages

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='서울은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 218, 'total_tokens': 243, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DuulLGeCATFx7tinQfmMUzfmqYFXN', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f02b6-61d5-7223-ab74-8cf28696a073-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '서울'}, 'id': 'call_2V0oLbCBcLr1iSbQ3BNLcq9S', 'type': 'tool_call'}], invalid

In [47]:
response.tool_calls

[{'name': 'get_current_time',
  'args': {'timezone': 'Asia/Seoul', 'location': '서울'},
  'id': 'call_2V0oLbCBcLr1iSbQ3BNLcq9S',
  'type': 'tool_call'}]

In [48]:
tool_call = response.tool_calls[0]
tool_call["name"]

'get_current_time'

In [49]:
tool_dict[tool_call["name"]]

StructuredTool(name='get_current_time', description="현재 시각을 반환하는 함수 \n\nArgs:\n    timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함\n    location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨", args_schema=<class 'langchain_core.utils.pydantic.get_current_time'>, func=<function get_current_time at 0x000002369D2C2340>)

In [50]:
tool_dict[tool_call["name"]].invoke(tool_call)

Asia/Seoul (서울) 현재시각 2026-06-26 15:55:35 


ToolMessage(content='Asia/Seoul (서울) 현재시각 2026-06-26 15:55:35 ', name='get_current_time', tool_call_id='call_2V0oLbCBcLr1iSbQ3BNLcq9S')

In [51]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # (7) tool_dict를 사용하여 도구 함수를 선택
    print(tool_call["args"]) # (8) 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # (9) 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

messages

{'timezone': 'Asia/Seoul', 'location': '서울'}
Asia/Seoul (서울) 현재시각 2026-06-26 15:55:35 


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='서울은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 218, 'total_tokens': 243, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DuulLGeCATFx7tinQfmMUzfmqYFXN', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f02b6-61d5-7223-ab74-8cf28696a073-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '서울'}, 'id': 'call_2V0oLbCBcLr1iSbQ3BNLcq9S', 'type': 'tool_call'}], invalid

In [52]:
llm_with_tools.invoke(messages)

AIMessage(content='서울(Asia/Seoul) 지금은 **15:55**입니다. (2026-06-26 15:55:35)', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 279, 'total_tokens': 312, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DuulM9nfHo64whiolKwtUNqsEQGME', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f02b6-65b7-7882-acb0-27c67354bf21-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 279, 'output_tokens': 33, 'total_tokens': 312, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [53]:
messages.append(llm_with_tools.invoke(messages))

In [54]:
messages

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='서울은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 218, 'total_tokens': 243, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DuulLGeCATFx7tinQfmMUzfmqYFXN', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f02b6-61d5-7223-ab74-8cf28696a073-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '서울'}, 'id': 'call_2V0oLbCBcLr1iSbQ3BNLcq9S', 'type': 'tool_call'}], invalid

### 주가 예제

In [55]:
import yfinance as yf

@tool
def get_yf_stock_history(ticker: str, period: str) -> str:
    """ 
    주식 종목의 가격 데이터를 조회하는 함수

    Args:
        ticker (str): 주식 종목 코드 (예: AAPL)
        period (str): 주식 데이터 조회 기간 (예: 1d, 1mo, 1y)
    
    """
    stock = yf.Ticker(ticker)
    history = stock.history(period=period)
    history_md = history.to_markdown() 

    return history_md

tools = [get_current_time, get_yf_stock_history]
tool_dict = {"get_current_time": get_current_time, "get_yf_stock_history": get_yf_stock_history}

llm_with_tools = llm.bind_tools(tools)

In [56]:
messages.append(HumanMessage("테슬라는 한달 전에 비해 주가가 올랐나 내렸나?"))

response = llm_with_tools.invoke(messages)
messages.append(response)

In [57]:
response.tool_calls

[{'name': 'get_yf_stock_history',
  'args': {'ticker': 'TSLA', 'period': '1mo'},
  'id': 'call_r6wqyUYtantXUGM65eZkaD6W',
  'type': 'tool_call'}]

In [58]:
#### 이어서 자연어로 모델이 답변하는 것 추가 실습

## 실습 1. pdf_summary.py

pdf_summary 하는 함수 tool로 등록하여 사용해보기

+ summarize_text 도 langchain 사용하는 것으로 변경

### (1) 수동 tool 루프 — Cell 6~13과 동일 패턴

In [81]:
import importlib
import os

import pdf_summary
importlib.reload(pdf_summary)
from langchain_core.messages import HumanMessage
from pdf_summary import summarize_pdf, extract_text_from_pdf

tools = [get_current_time, get_yf_stock_history, summarize_pdf, extract_text_from_pdf]
tool_dict = {
    "get_current_time": get_current_time,
    "get_yf_stock_history": get_yf_stock_history,
    "summarize_pdf": summarize_pdf,
    "extract_text_from_pdf": extract_text_from_pdf,
}

llm_with_tools = llm.bind_tools(tools)

pdf_path = os.path.join(os.getcwd(), "samples/videoanomaly.pdf")

messages = [
    HumanMessage(f"이 {pdf_path} 문서의 저자 목록과 간단한 논문 요약을 출력해줘?"),
]

response = llm_with_tools.invoke(messages)
messages.append(response)

# 1차 응답이 tool_calls면 도구 실행 후 2차 invoke
if response.tool_calls:
    for tool_call in response.tool_calls:
        tool_msg = tool_dict[tool_call["name"]].invoke(tool_call)
        messages.append(tool_msg)
    response_final = llm_with_tools.invoke(messages)
    messages.append(response_final)

print(response_final.content)


content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 419, 'total_tokens': 465, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DuuwRqkikK3yBdpZeYKUqpCDMPwdm', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019f02c0-e078-79e1-b962-10b690fef4be-0' tool_calls=[{'name': 'summarize_pdf', 'args': {'pdf_path': 'd:\\autornd\\SK Autonomous R&D\\실습\\5일차\\samples\\videoanomaly.pdf', 'save': False}, 'id': 'call_wDpaxAGNjE6Z5I3HVwQzsCKS', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 419, 'output_tokens': 46, 'total_tokens': 465, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_to

In [86]:
response.tool_calls

[{'name': 'summarize_pdf',
  'args': {'pdf_path': 'd:\\autornd\\SK Autonomous R&D\\실습\\5일차\\samples\\videoanomaly.pdf',
   'save': False},
  'id': 'call_wDpaxAGNjE6Z5I3HVwQzsCKS',
  'type': 'tool_call'}]

In [84]:
print(response_final.content)

아래는 `videoanomaly.pdf` 문서의 **저자 목록**과 **간단 논문 요약**입니다.

## 1) 저자 목록
- Qianzi Yu  
- Kai Zhu  
- Yang Cao  
- Yu Kang  

## 2) 간단 논문 요약
- 저자들은 산업 현장의 비디오 이상 탐지에서 기존 방법이 **국소/미세 특징** 및 **반복적인 제품 이동(등간격 주기성)**을 충분히 활용하지 못한다고 지적합니다. 또한 산업 영상은 **조명 변화**에 의해 성능이 흔들리기 쉽다고 봅니다.
- 이를 해결하기 위해 **주파수 우선화 기반 이상 탐지 모델 INFP**를 제안합니다. 핵심은 움직이는 제품과 정적 배경이 **주파수 영역에서 구분되는 특성**을 학습에 활용하는 것입니다.
- 구체적으로는 고정된 이동 궤적에 해당하는 **고주파 에너지**에 더 높은 가중치를 주는 **trajectory filter**를 설계하고, **주파수 특징과 시공간 특징을 결합하는 multi-feature fusion** 모듈을 통해 등간격 주기성을 더 잘 반영하도록 합니다.
- 실험에서는 **IPAD 데이터셋**에서 제안 방법이 최신 기법들보다 **우수한 성능**을 보였으며, **강건성(robustness)** 및 **일반화 성능**이 개선됨을 주장합니다.


### (2) AgentExecutor로 추상화

Cell 6~13의 `tool_calls` 루프를 **`AgentExecutor`** 가 내부에서 자동 처리합니다. (5.4에서도 동일 패턴)

In [87]:
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

tools = [get_current_time, get_yf_stock_history, summarize_pdf, extract_text_from_pdf]

prompt = ChatPromptTemplate.from_messages([
    ("system", "로컬 PDF 경로가 주어지면 반드시 도구를 사용해 내용을 읽고 한국어로 답하세요."),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])
                               
agent = create_tool_calling_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True, max_iterations=5)

pdf_path = os.path.join(os.getcwd(), "samples/videoanomaly.pdf")
result = executor.invoke({
    "input": f"이 {pdf_path} 문서의 저자 목록과 간단한 논문 요약을 출력해줘"
})

print(result["output"])



> Entering new AgentExecutor chain...

Invoking: `summarize_pdf` with `{'pdf_path': 'd:\\autornd\\SK Autonomous R&D\\실습\\5일차\\samples\\videoanomaly.pdf', 'save': False}`


# INFP: 산업용 비디오 이상 탐지를 위한 주파수 우선화 방법

## 저자의 문제 인식 및 주장 (15문장 이내)

저자들은 기존 비디오 이상 탐지 방법이 산업용 생산 라인 영상에는 잘 맞지 않는다고 본다.  
기존 방법은 여러 정상 객체가 넓게 분포한 일반 영상에 맞춰져 있어, 산업 영상처럼 단일 제품이 좁은 영역을 따라 움직이는 상황을 충분히 반영하지 못한다.  
산업 영상은 제품이 화면에서 희소하게 나타나고, 고정된 궤적을 따라 이동하며, 일정한 시간 간격의 반복 패턴을 가진다.  
또한 조명 변화의 영향이 커서 탐지 성능이 쉽게 흔들린다.  
이를 해결하기 위해 저자들은 INFP라는 인코더-디코더 기반 프레임워크를 제안한다.  
핵심 아이디어는 영상의 주파수 도메인 특징을 활용해 제품의 주기성과 동적 특성을 더 잘 포착하는 것이다.  
먼저 trajectory filter를 통해 움직이는 객체와 정적 배경의 주파수 차이를 이용해 고정 이동 경로에 더 큰 가중치를 준다.  
이를 통해 제품 영역에 더 집중하고 배경의 방해를 줄인다.  
또한 multi-feature fusion module을 통해 주파수 특징과 시공간 특징을 결합한다.  
이 과정에서 산업 영상의 등간격 주기성을 활용해 속도 이상이나 제품 누락 같은 문제를 더 잘 잡아낸다.  
결합된 특징에 문맥 정보를 추가로 통합해 조명 변화로 인한 픽셀 변동의 영향을 완화한다.  
실험 결과, IPAD 데이터셋에서 제안 방법이 기존 최신 방법보다 우수한 성능과 더 나은 강건성을 보였다.  

## 저자 소개

이 논문의 저자는 Qianzi Yu, Kai Zhu, Yang Cao, Yu Kan